<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-22</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>CLIP 평가(CLIP Evaluation)</strong>에 대해 학습합니다.</br></br>
>CLIP 모델로 생성 이미지와 텍스트 간의 유사도를 학습해봅시다.

<div style="text-align:center">

</div>

</br>

# CLIP (Contrastive Language-Image Pre-training)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트와 이미지를 동일한 임베딩 공간</mark>에 매핑하여 유사도를 계산하는 모델입니다.

> 기존 비전 모델은 사전에 정의된 카테고리만 분류할 수 있어 새 카테고리를 추가하려면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">재학습</mark>이 필요합니다.</br></br>
 > CLIP은 이미지와 텍스트를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 임베딩 공간에 매핑</mark>하여 두 모달리티 간 코사인 유사도를 직접 비교함으로써 이 한계를 극복합니다.</br></br>
 > 분류하고 싶은 카테고리를 텍스트로 서술하기만 하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">별도의 학습 없이</mark> 이미지가 어떤 텍스트 설명과 가장 유사한지 계산하여 분류할 수 있으며, 새로운 카테고리도 텍스트 작성만으로 즉시 대응 가능하다는 점이 제로샷 분류의 핵심 혁신입니다.</br>

<div style="text-align:center">

</div>

<div style="text-align:center">

</div>

</br>

## CLIPProcessor & CLIPModel

In [4]:
# TODO 1: CLIP 모델과 프로세서를 사전학습 모델로부터 로드해봅시다.
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [10]:
from PIL import Image

# TODO 2: 이미지와 여러 텍스트 설명을 전처리하여 입력을 준비해봅시다.
image = Image.open('dog.jpg')
inputs = processor(
    text=['a dog', 'a cat', 'a car'],
    images=image,
    return_tensors='pt',  # 파이토치의 텐서로 변경
    padding=True
)

</br>

## 유사도 계산

In [11]:
# TODO 3: 모델로 추론해봅시다.
import torch

with torch.no_grad():
    outputs = model(**inputs)

outputs.keys()


odict_keys(['logits_per_image', 'logits_per_text', 'text_embeds', 'image_embeds', 'text_model_output', 'vision_model_output'])

💡CLIP 모델 출력 키

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">키</th>
      <th>설명</th>
      <th style="text-align:center">Shape 예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>logits_per_image</code></td><td>이미지 기준 유사도 스코어 (이미지 1개 vs 텍스트 N개)</td><td style="text-align:center">(1, 3)</td></tr>
    <tr><td style="text-align:center"><code>logits_per_text</code></td><td>텍스트 기준 유사도 스코어 (logits_per_image의 전치)</td><td style="text-align:center">(3, 1)</td></tr>
    <tr><td style="text-align:center"><code>text_embeds</code></td><td>각 텍스트가 변환된 임베딩 벡터</td><td style="text-align:center">(3, 512)</td></tr>
    <tr><td style="text-align:center"><code>image_embeds</code></td><td>이미지가 변환된 임베딩 벡터</td><td style="text-align:center">(1, 512)</td></tr>
    <tr><td style="text-align:center"><code>text_model_output</code></td><td>텍스트 인코더의 전체 출력 (중간 hidden states 포함)</td><td style="text-align:center">-</td></tr>
    <tr><td style="text-align:center"><code>vision_model_output</code></td><td>이미지 인코더의 전체 출력 (중간 hidden states 포함)</td><td style="text-align:center">-</td></tr>
  </tbody>
</table>

> 주로 사용하는 것은 상위 4개이며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">logits_per_image</mark>로 유사도를 비교하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">text_embeds/image_embeds</mark>로 임베딩을 직접 다룹니다.

In [12]:
# TODO 4: logits_per_image에서 유사도 스코어를 확인해봅시다.
logits_per_image = outputs.logits_per_image
logits_per_image


tensor([[25.7711, 20.4452, 19.1892]])

💡Logit이란?
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Logit</mark>은 모델이 출력하는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정규화되지 않은 원시 점수</mark>입니다.</br></br>
> 아직 확률이 아니라 단순한 숫자이므로, 값 자체보다는 **상대적 크기**가 중요합니다.</br>
> 예: `[18.5, 12.3, 5.1]` → "a dog"이 가장 높은 점수 → 이 이미지는 강아지에 가장 가까움</br></br>
> 이 점수를 확률로 변환하려면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">소프트맥스(Softmax)</mark>를 적용합니다.</br>
> `[18.5, 12.3, 5.1]` → softmax → `[0.998, 0.002, 0.000]` (합 = 1.0)

In [13]:
# TODO 5: 유사도 스코어에 소프트맥스를 적용하여 확률로 변환해봅시다.
value = logits_per_image.softmax(dim=1)
value


tensor([[0.9938, 0.0048, 0.0014]])

In [15]:
# TODO 6: 각 텍스트 라벨별 유사도 확률을 출력해봅시다.
labels = ['a dog', 'a cat', 'a car']
for label, v in zip(labels, value[0]):
    print(f"{label}: {v:.4f}")

a dog:0.9937893748283386
a cat:0.004833914805203676
a car:0.0013766753254458308


💡logits_per_image vs logits_per_text
> `logits_per_image`: 하나의 이미지 vs 여러 텍스트 (이미지 기준)</br>
> `logits_per_text`: 하나의 텍스트 vs 여러 이미지 (텍스트 기준)

</br>

## Zero-shot 분류
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">라벨 학습 없이</mark> 텍스트 설명만으로 이미지를 분류합니다.

In [ ]:
# TODO 7: 여러 텍스트 라벨을 정의하고, 이미지와 함께 전처리 후 모델로 추론하여 가장 유사도가 높은 라벨을 Zero-shot 분류 결과로 출력해봅시다.
labels = ['a dog', 'a cat', 'a car', 'a bird', 'a horse']

inputs = processor(
    text=labels,
    images=image,
    return_tensors='pt',
    padding=True
)

with torch.no_grad():
    outputs = model(**inputs)

probs = outputs.logits_per_image.softmax(dim=1)
pred_idx = probs.argmax(dim=1).item()

print("Zero-shot 분류 결과:", labels[pred_idx])
print("확률:", probs[0][pred_idx].item())

</br>

## 생성 이미지 품질 평가
> 생성된 이미지가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">원본 프롬프트와 얼마나 일치</mark>하는지 CLIP Score로 평가합니다.

In [17]:
# TODO 8: 이미지와 텍스트를 입력받아 CLIP 유사도 점수를 반환하는 함수를 정의하고, 생성된 이미지와 프롬프트 간의 CLIP Score를 계산해봅시다.
def clip_score(image, text, model, processor):
    inputs = processor(
        text=[text],
        images=image,
        return_tensors='pt',
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits_per_image.item()

score1 = clip_score(image, 'a golden retriever puppy', model, processor)
score2 = clip_score(image, 'a cat sitting on a windowsill', model, processor)

print(score1, score2)

32.59756851196289 15.02584171295166


💡CLIP Score가 높으면 좋은 이미지인가?
> CLIP Score는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트-이미지 정합도</mark>만 측정합니다.</br>
> 이미지 품질(해상도, 아티팩트)은 별도의 지표(FID, IS)로 평가해야 합니다.